## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 13: Hyperparameter-Optimierung
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print("=== Optuna Hyperparameter-Suche ===\n")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_all = x_train_all.astype('float32') / 255.0
x_test      = x_test.astype('float32') / 255.0
x_train_all = x_train_all.reshape(-1, 784)
x_test      = x_test.reshape(-1, 784)
x_train = x_train_all[:5000]
y_train = y_train_all[:5000]

# ── 2. Optuna importieren oder manuell implementieren ──────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    USE_OPTUNA = True
    print("Optuna gefunden! Verwende echten TPE-Sampler.\n")
except ImportError:
    USE_OPTUNA = False
    print("Optuna nicht installiert. Implementiere TPE-Simulation manuell.\n")

# ── 3. Objective-Funktion (kompatibel mit Optuna und Simulation)
class TrialSimulator:
    """Simuliert optuna.Trial.suggest_* für manuelle Implementierung."""
    def __init__(self, **params):
        self._params = params

    def suggest_float(self, name, low, high, log=False):
        return self._params.get(name, low)

    def suggest_int(self, name, low, high):
        return self._params.get(name, low)

    def suggest_categorical(self, name, choices):
        return self._params.get(name, choices[0])

def objective_fn(trial):
    """Objective-Funktion: optimiert 5 Hyperparameter."""
    # Hyperparameter mit trial.suggest_*
    n_layers    = trial.suggest_int('n_layers', 1, 3)
    n_units     = trial.suggest_int('n_units', 32, 256)
    dropout     = trial.suggest_float('dropout', 0.0, 0.5)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [32, 64, 128])

    # Modell bauen
    layers_list = [layers.Dense(n_units, activation='relu', input_shape=(784,)),
                   layers.Dropout(dropout)]
    for _ in range(n_layers - 1):
        layers_list.append(layers.Dense(max(n_units//2, 16), activation='relu'))
        layers_list.append(layers.Dropout(dropout/2))
    layers_list.append(layers.Dense(10, activation='softmax'))

    model = keras.Sequential(layers_list)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    hist = model.fit(x_train, y_train, epochs=10, batch_size=batch_size,
                      validation_split=0.2, verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(patience=3)])
    val_acc = max(hist.history['val_accuracy'])
    return val_acc

# ── 4a. Optuna Study ──────────────────────────────────────
if USE_OPTUNA:
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    N_TRIALS = 30
    print(f"Starte Optuna-Studie: {N_TRIALS} Trials, TPE-Sampler...")

    trial_values = []
    best_so_far  = []

    def callback(study, trial):
        trial_values.append(trial.value)
        best_so_far.append(study.best_value)
        print(f"  Trial {trial.number+1:2d}: {trial.value:.4f} | "
              f"Best: {study.best_value:.4f} | "
              f"Params: {trial.params}")

    study.optimize(objective_fn, n_trials=N_TRIALS, callbacks=[callback])

    best_params = study.best_params
    best_val_acc = study.best_value
    all_trial_vals = trial_values

    print(f"\nBeste Hyperparameter:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"Beste Val-Accuracy: {best_val_acc:.4f}")

    # Importances
    try:
        importances = optuna.importance.get_param_importances(study)
        print(f"\nParameter-Importances: {importances}")
    except Exception:
        importances = None

# ── 4b. Manuelle TPE-Simulation ───────────────────────────
else:
    # Vereinfachte TPE-Simulation: erst random, dann guided
    N_TRIALS    = 30
    N_RANDOM    = 10  # Zufällige Startpunkte
    all_trials  = []
    trial_values = []
    best_so_far  = []
    best_val_acc = 0.0
    best_params  = {}

    def sample_from_posterior(all_trials, n_top=5):
        """Vereinfachte TPE: Sample aus den Top-k Ergebnissen."""
        if len(all_trials) < N_RANDOM:
            return None   # Random-Phase
        sorted_t = sorted(all_trials, key=lambda x: x['val'], reverse=True)
        good     = sorted_t[:n_top]
        # Perturb den besten
        base = good[np.random.randint(len(good))]['params']
        return {
            'lr':         np.clip(base['lr'] * np.exp(np.random.randn() * 0.3), 1e-4, 1e-2),
            'n_units':    int(np.clip(base['n_units'] + np.random.randint(-32, 33), 32, 256)),
            'dropout':    np.clip(base['dropout'] + np.random.randn() * 0.05, 0.0, 0.5),
            'n_layers':   np.random.choice([1, 2, 3], p=[0.2, 0.5, 0.3]),
            'batch_size': np.random.choice([32, 64, 128])
        }

    print(f"Manuelle TPE-Simulation: {N_TRIALS} Trials...")
    for t in range(N_TRIALS):
        # Parameter samplen
        suggested = sample_from_posterior(all_trials)
        if suggested is None:
            params = {
                'lr':         10 ** np.random.uniform(-4, -2),
                'n_units':    int(np.random.choice([32, 64, 96, 128, 192, 256])),
                'dropout':    np.random.uniform(0, 0.5),
                'n_layers':   np.random.choice([1, 2, 3]),
                'batch_size': int(np.random.choice([32, 64, 128]))
            }
        else:
            params = suggested

        trial = TrialSimulator(**params)
        val = objective_fn(trial)
        all_trials.append({'params': params, 'val': val})
        trial_values.append(val)

        if val > best_val_acc:
            best_val_acc = val
            best_params  = params.copy()
        best_so_far.append(best_val_acc)

        print(f"  Trial {t+1:2d}: {val:.4f} | Best: {best_val_acc:.4f} | "
              f"lr={params['lr']:.5f}, units={params['n_units']}, drop={params['dropout']:.2f}")

    importances = None
    all_trial_vals = trial_values
    print(f"\nBeste Hyperparameter: {best_params}")

# ── 5. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Optuna Hyperparameter-Optimierung', fontsize=14)

# Konvergenz
ax = axes[0, 0]
ax.plot(all_trial_vals, 'o', color='blue', alpha=0.5, label='Trial Wert')
ax.plot(best_so_far,    color='red', linewidth=2, label='Bestes bisher')
ax.set_title('Konvergenz')
ax.set_xlabel('Trial'); ax.set_ylabel('Val-Accuracy')
ax.legend(); ax.grid(True)

# Scatter: lr vs. Val-Acc
ax2 = axes[0, 1]
if USE_OPTUNA:
    lrs  = [t.params['lr']    for t in study.trials]
    vals = [t.value           for t in study.trials]
else:
    lrs  = [t['params']['lr'] for t in all_trials]
    vals = [t['val']          for t in all_trials]
scatter = ax2.scatter(lrs, vals, c=range(len(vals)), cmap='viridis', s=60)
plt.colorbar(scatter, ax=ax2, label='Trial-Nummer')
ax2.set_xscale('log')
ax2.set_title('LR vs. Val-Accuracy (Farbe=Zeit)')
ax2.set_xlabel('LR (log)'); ax2.set_ylabel('Val-Acc'); ax2.grid(True)

# Parameter Importances (wenn verfügbar)
ax3 = axes[1, 0]
if importances:
    ax3.barh(list(importances.keys()), list(importances.values()), color='steelblue')
    ax3.set_title('Parameter-Wichtigkeit (Optuna)')
    ax3.set_xlabel('Wichtigkeit'); ax3.grid(True, axis='x')
else:
    # Manuelle Korrelation
    if not USE_OPTUNA:
        param_names = list(all_trials[0]['params'].keys())
        corrs = []
        for pn in param_names:
            p_vals  = [t['params'].get(pn, 0) for t in all_trials]
            if len(set(p_vals)) > 1:
                c = np.corrcoef(p_vals, [t['val'] for t in all_trials])[0, 1]
            else:
                c = 0.0
            corrs.append(abs(c))
        ax3.barh(param_names, corrs, color='steelblue')
        ax3.set_title('Parameter-Korrelation mit Val-Accuracy')
        ax3.set_xlabel('|Korrelation|'); ax3.grid(True, axis='x')

# Top-10 Trials
ax4 = axes[1, 1]
sorted_vals = sorted(enumerate(all_trial_vals), key=lambda x: x[1], reverse=True)[:10]
trial_idxs  = [f"T{x[0]+1}" for x in sorted_vals]
top_vals    = [x[1] for x in sorted_vals]
ax4.bar(range(10), top_vals, color='steelblue', alpha=0.8)
ax4.set_xticks(range(10)); ax4.set_xticklabels(trial_idxs, fontsize=8)
ax4.set_title('Top-10 Trials'); ax4.set_ylabel('Val-Accuracy')
ax4.axhline(best_val_acc, color='red', linestyle='--', label=f'Best: {best_val_acc:.4f}')
ax4.legend(); ax4.grid(True, axis='y')

plt.tight_layout()
plt.savefig('E13_1_optuna_hyperparameter.png', dpi=100)
plt.show()

print(f"\nBeste Val-Accuracy: {best_val_acc:.4f}")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 13: Hyperparameter-Optimierung
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import itertools

np.random.seed(42)
tf.random.set_seed(42)

print("=== Neural Architecture Search (NAS) vereinfacht ===\n")
print("Suche über: n_layers [1-5], n_units, activation")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_all = x_train_all.astype('float32') / 255.0
x_test      = x_test.astype('float32') / 255.0
x_train_all = x_train_all.reshape(-1, 784)
x_test      = x_test.reshape(-1, 784)
x_train = x_train_all[:4000]
y_train = y_train_all[:4000]

# ── 2. Architektur-Suchraum definieren ────────────────────
ARCH_SPACE = {
    'n_layers':   [1, 2, 3, 4, 5],
    'n_units':    [32, 64, 128, 256],
    'activation': ['relu', 'tanh', 'elu']
}

def encode_arch(n_layers, n_units, activation):
    """Kodiert Architektur als Tuple für Dictionary-Key."""
    return (n_layers, n_units, activation)

def build_arch(n_layers, n_units, activation):
    """Baut Netzwerk mit gegebener Architektur."""
    layer_list = []
    # Erste Schicht: Eingabe
    layer_list.append(layers.Dense(n_units, activation=activation, input_shape=(784,)))
    # Versteckte Schichten (progressiv kleiner)
    for i in range(1, n_layers):
        curr_units = max(n_units // (2 ** i), 16)
        layer_list.append(layers.Dense(curr_units, activation=activation))
    # Ausgabeschicht
    layer_list.append(layers.Dense(10, activation='softmax'))

    model = keras.Sequential(layer_list)
    n_params = model.count_params()
    return model, n_params

def evaluate_arch(n_layers, n_units, activation):
    """Evaluiert eine Architektur, gibt Val-Acc und Parameter-Anzahl zurück."""
    model, n_params = build_arch(n_layers, n_units, activation)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_train, y_train, epochs=10, batch_size=64,
                      validation_split=0.2, verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(patience=3)])
    val_acc = max(hist.history['val_accuracy'])
    return val_acc, n_params

# ── 3. Random Search über 50 Architekturen ────────────────
N_CONFIGS = 50
results   = []

print(f"Evaluiere {N_CONFIGS} zufällige Architekturen...\n")

# Alle möglichen Kombinationen berechnen
all_archs = list(itertools.product(
    ARCH_SPACE['n_layers'],
    ARCH_SPACE['n_units'],
    ARCH_SPACE['activation']
))
print(f"Gesamter Suchraum: {len(all_archs)} Architekturen")
print(f"Zufällige Stichprobe: {N_CONFIGS} davon\n")

# Zufällig auswählen (ohne Wiederholung)
selected_archs = [all_archs[i] for i in
                   np.random.choice(len(all_archs), min(N_CONFIGS, len(all_archs)), replace=False)]

for i, (n_layers, n_units, activation) in enumerate(selected_archs):
    val_acc, n_params = evaluate_arch(n_layers, n_units, activation)
    arch_str = f"{n_layers}L-{n_units}u-{activation}"
    results.append({
        'n_layers': n_layers, 'n_units': n_units, 'activation': activation,
        'val_acc': val_acc, 'n_params': n_params, 'arch_str': arch_str
    })
    print(f"[{i+1:2d}/{N_CONFIGS}] {arch_str:20s} | Val-Acc={val_acc:.4f}, Params={n_params:,}")

# ── 4. Ergebnisse analysieren ─────────────────────────────
results.sort(key=lambda x: x['val_acc'], reverse=True)
best = results[0]

print(f"\nBeste Architektur:")
print(f"  {best['n_layers']} Layer × {best['n_units']} Units, {best['activation']}")
print(f"  Val-Acc: {best['val_acc']:.4f}, Parameter: {best['n_params']:,}")

# ── 5. Analyse: Effiziente Frontier ───────────────────────
# Welche Architekturen haben das beste Acc/Parameter-Verhältnis?
print("\nTop-5 nach Val-Accuracy:")
for i, r in enumerate(results[:5], 1):
    eff = r['val_acc'] / (np.log10(r['n_params']) + 1)
    print(f"  {i}. {r['arch_str']:20s} | Acc={r['val_acc']:.4f} | "
          f"Params={r['n_params']:,} | Effizienz={eff:.4f}")

# ── 6. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Neural Architecture Search (NAS): Architekturraum', fontsize=13)

# 1. Acc vs. Parameter (Scatterplot)
ax = axes[0, 0]
for act in ARCH_SPACE['activation']:
    sub = [r for r in results if r['activation'] == act]
    ax.scatter([r['n_params'] for r in sub], [r['val_acc'] for r in sub],
                label=act, alpha=0.7, s=50)
ax.set_xscale('log')
ax.set_title('Parameter vs. Val-Accuracy')
ax.set_xlabel('Parameter (log)'); ax.set_ylabel('Val-Accuracy')
ax.legend(); ax.grid(True)

# 2. n_layers vs. Val-Acc (Boxplot)
ax2 = axes[0, 1]
layers_vals = [ARCH_SPACE['n_layers']]
box_data = [[r['val_acc'] for r in results if r['n_layers'] == l]
             for l in ARCH_SPACE['n_layers']]
ax2.boxplot(box_data, labels=ARCH_SPACE['n_layers'])
ax2.set_title('Val-Acc nach Anzahl Schichten')
ax2.set_xlabel('n_layers'); ax2.set_ylabel('Val-Accuracy'); ax2.grid(True)

# 3. n_units vs. Val-Acc (Boxplot)
ax3 = axes[0, 2]
box_data2 = [[r['val_acc'] for r in results if r['n_units'] == u]
              for u in ARCH_SPACE['n_units']]
ax3.boxplot(box_data2, labels=ARCH_SPACE['n_units'])
ax3.set_title('Val-Acc nach Units pro Schicht')
ax3.set_xlabel('n_units'); ax3.set_ylabel('Val-Accuracy'); ax3.grid(True)

# 4. Aktivierungsfunktion vs. Val-Acc
ax4 = axes[1, 0]
box_data3 = [[r['val_acc'] for r in results if r['activation'] == a]
              for a in ARCH_SPACE['activation']]
ax4.boxplot(box_data3, labels=ARCH_SPACE['activation'])
ax4.set_title('Val-Acc nach Aktivierungsfunktion')
ax4.set_xlabel('Aktivierung'); ax4.set_ylabel('Val-Accuracy'); ax4.grid(True)

# 5. 2D Heatmap: n_layers × n_units (beste Acc pro Kombination)
ax5 = axes[1, 1]
heatmap = np.zeros((len(ARCH_SPACE['n_layers']), len(ARCH_SPACE['n_units'])))
for r in results:
    i = ARCH_SPACE['n_layers'].index(r['n_layers'])
    j = ARCH_SPACE['n_units'].index(r['n_units'])
    if heatmap[i, j] < r['val_acc']:
        heatmap[i, j] = r['val_acc']
im = ax5.imshow(heatmap, cmap='YlOrRd', aspect='auto')
ax5.set_xticks(range(len(ARCH_SPACE['n_units']))); ax5.set_xticklabels(ARCH_SPACE['n_units'])
ax5.set_yticks(range(len(ARCH_SPACE['n_layers']))); ax5.set_yticklabels(ARCH_SPACE['n_layers'])
ax5.set_xlabel('n_units'); ax5.set_ylabel('n_layers')
ax5.set_title('Beste Val-Acc: n_layers × n_units')
plt.colorbar(im, ax=ax5)
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        ax5.text(j, i, f'{heatmap[i,j]:.3f}', ha='center', va='center', fontsize=7)

# 6. Top-10 Architekturen
ax6 = axes[1, 2]
top10 = results[:10]
ax6.barh(range(10), [r['val_acc'] for r in top10], color='steelblue', alpha=0.8)
ax6.set_yticks(range(10))
ax6.set_yticklabels([r['arch_str'] for r in top10], fontsize=8)
ax6.set_title('Top-10 Architekturen')
ax6.set_xlabel('Val-Accuracy'); ax6.grid(True, axis='x')

plt.tight_layout()
plt.savefig('E13_2_nas_architekturraum.png', dpi=100)
plt.show()

print("\nNAS abgeschlossen!")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 13: Hyperparameter-Optimierung
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import copy

np.random.seed(42)
tf.random.set_seed(42)

print("=== Population Based Training (PBT) Simulation ===\n")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_all = x_train_all.astype('float32') / 255.0
x_test      = x_test.astype('float32') / 255.0
x_train_all = x_train_all.reshape(-1, 784)
x_test      = x_test.reshape(-1, 784)
x_train = x_train_all[:4000]
y_train = y_train_all[:4000]

# ── 2. PBT-Konfiguration ──────────────────────────────────
N_POPULATION  = 10     # Anzahl paralleler Modelle
TOTAL_EPOCHS  = 30     # Gesamtepochen
EXPLOIT_EVERY = 10     # Alle X Epochen Exploit+Explore
PERTURBATION  = 0.2    # Hyperparameter-Störung (±20%)
N_BATCHES_PER_EPOCH = 50   # Batches pro Epoche (für Speed)

def create_random_individual():
    """Erstellt ein Individuum mit zufälligen HP."""
    return {
        'lr':       10 ** np.random.uniform(-3.5, -1.5),
        'units':    int(np.random.choice([32, 64, 128, 256])),
        'dropout':  np.random.uniform(0, 0.4),
        'momentum': np.random.uniform(0.8, 0.99)
    }

def build_model_for_individual(ind):
    """Baut und kompiliert Modell für gegebenes Individuum."""
    model = keras.Sequential([
        layers.Dense(ind['units'], activation='relu', input_shape=(784,)),
        layers.Dropout(ind['dropout']),
        layers.Dense(max(ind['units']//2, 16), activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=ind['lr'], momentum=ind['momentum']),
        loss='sparse_categorical_crossentropy', metrics=['accuracy']
    )
    return model

# ── 3. Population initialisieren ──────────────────────────
print("Initialisiere Population...")
population = []
for i in range(N_POPULATION):
    hps = create_random_individual()
    model = build_model_for_individual(hps)
    population.append({
        'id': i,
        'model': model,
        'hps': hps,
        'val_acc_history': [],
        'exploit_count': 0,
        'hp_history': [copy.deepcopy(hps)]
    })
    print(f"  Individuum {i}: lr={hps['lr']:.5f}, units={hps['units']}, "
          f"dropout={hps['dropout']:.2f}, momentum={hps['momentum']:.3f}")

# ── 4. PBT Training ───────────────────────────────────────
n_batches = len(x_train) // 64
print(f"\nStarte PBT: {N_POPULATION} Individuen, {TOTAL_EPOCHS} Epochen")
print(f"Exploit alle {EXPLOIT_EVERY} Epochen\n")

pbt_metrics = {i: {'val_acc': [], 'lr_history': []} for i in range(N_POPULATION)}
exploit_events = []

for epoch in range(1, TOTAL_EPOCHS + 1):
    # Jedes Individuum für eine Epoche trainieren
    for ind in population:
        idx    = np.random.permutation(len(x_train))[:N_BATCHES_PER_EPOCH * 64]
        x_ep   = x_train[idx]
        y_ep   = y_train[idx]
        ind['model'].fit(x_ep, y_ep, epochs=1, batch_size=64, verbose=0)

    # Validierung aller Individuen
    for ind in population:
        val_acc = ind['model'].evaluate(
            x_train_all[4000:5000], y_train_all[4000:5000], verbose=0)[1]
        ind['val_acc_history'].append(val_acc)
        pbt_metrics[ind['id']]['val_acc'].append(val_acc)
        pbt_metrics[ind['id']]['lr_history'].append(ind['hps']['lr'])

    # Exploit + Explore alle EXPLOIT_EVERY Epochen
    if epoch % EXPLOIT_EVERY == 0:
        print(f"\nEpoche {epoch}: EXPLOIT + EXPLORE")
        # Sortiere nach Val-Acc
        sorted_pop = sorted(population, key=lambda x: x['val_acc_history'][-1], reverse=True)
        top_half   = sorted_pop[:N_POPULATION//2]
        bot_half   = sorted_pop[N_POPULATION//2:]

        print(f"  Top-5 (behalten): {[f'{p[\"id\"]}={p[\"val_acc_history\"][-1]:.4f}' for p in top_half]}")
        print(f"  Bot-5 (ersetzen): {[f'{p[\"id\"]}={p[\"val_acc_history\"][-1]:.4f}' for p in bot_half]}")

        for bot_ind in bot_half:
            # EXPLOIT: Kopiere Gewichte vom zufälligen Top-Individuum
            source = top_half[np.random.randint(len(top_half))]
            bot_ind['model'].set_weights(source['model'].get_weights())
            bot_ind['exploit_count'] += 1

            # EXPLORE: Störe Hyperparameter des kopierten Individuums
            new_hps = copy.deepcopy(source['hps'])
            new_hps['lr']       = np.clip(new_hps['lr'] * (1 + np.random.uniform(-PERTURBATION, PERTURBATION)), 1e-5, 0.1)
            new_hps['dropout']  = np.clip(new_hps['dropout'] + np.random.uniform(-0.1, 0.1), 0.0, 0.5)
            new_hps['momentum'] = np.clip(new_hps['momentum'] + np.random.uniform(-0.05, 0.05), 0.7, 0.99)

            # Neuen Optimierer mit geänderten HP setzen
            bot_ind['hps'] = new_hps
            bot_ind['model'].compile(
                optimizer=keras.optimizers.SGD(
                    learning_rate=new_hps['lr'], momentum=new_hps['momentum']),
                loss='sparse_categorical_crossentropy', metrics=['accuracy']
            )
            bot_ind['hp_history'].append(copy.deepcopy(new_hps))
            exploit_events.append(epoch)

    # Status ausgeben
    val_accs_now = [p['val_acc_history'][-1] for p in population]
    print(f"Epoche {epoch:2d}: Mean={np.mean(val_accs_now):.4f}, "
          f"Best={max(val_accs_now):.4f}, Worst={min(val_accs_now):.4f}")

# ── 5. Vergleich mit Random Search ────────────────────────
print("\n\nVergleich: PBT vs. Random Search")
print("Trainiere 10 zufällige Modelle mit fixierten HPs...")

random_results = []
for i in range(N_POPULATION):
    hps = create_random_individual()
    model = build_model_for_individual(hps)
    # Gleiche Anzahl Epochen
    n_train = len(x_train)
    model.fit(x_train, y_train, epochs=TOTAL_EPOCHS,
               batch_size=64, verbose=0)
    val_acc = model.evaluate(x_train_all[4000:5000], y_train_all[4000:5000], verbose=0)[1]
    random_results.append(val_acc)
    print(f"  Random {i}: val_acc={val_acc:.4f}")

# ── 6. Ergebnisse ─────────────────────────────────────────
pbt_final    = [p['val_acc_history'][-1] for p in population]
pbt_best     = max(pbt_final)
pbt_mean     = np.mean(pbt_final)
random_best  = max(random_results)
random_mean  = np.mean(random_results)

print(f"\nPBT:          Best={pbt_best:.4f}, Mean={pbt_mean:.4f}")
print(f"Random Search: Best={random_best:.4f}, Mean={random_mean:.4f}")

# ── 7. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Population Based Training (PBT) vs. Random Search', fontsize=13)

# Lernkurven aller PBT-Individuen
ax = axes[0, 0]
colors = plt.cm.tab10(np.linspace(0, 1, N_POPULATION))
for i, (ind, color) in enumerate(zip(population, colors)):
    ax.plot(ind['val_acc_history'], color=color, alpha=0.7,
             label=f'Ind {ind["id"]} (final={ind["val_acc_history"][-1]:.3f})')
# Exploit-Ereignisse markieren
unique_exploits = list(set(exploit_events))
for ev in unique_exploits:
    ax.axvline(ev - 1, color='gray', linestyle=':', alpha=0.4)
ax.set_title('PBT: Val-Accuracy aller Individuen')
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-Accuracy')
ax.legend(fontsize=6, loc='lower right'); ax.grid(True)

# Mittlere Population-Performance über Zeit
ax2 = axes[0, 1]
all_accs_matrix = np.array([ind['val_acc_history'] for ind in population])
mean_per_epoch  = all_accs_matrix.mean(axis=0)
max_per_epoch   = all_accs_matrix.max(axis=0)
min_per_epoch   = all_accs_matrix.min(axis=0)
epochs_x = range(1, TOTAL_EPOCHS + 1)
ax2.plot(epochs_x, mean_per_epoch, color='blue', label='PBT Mittelwert', linewidth=2)
ax2.fill_between(epochs_x, min_per_epoch, max_per_epoch, alpha=0.2, color='blue')
ax2.axhline(random_mean, color='red', linestyle='--', label=f'Random Mean: {random_mean:.4f}')
ax2.axhline(random_best, color='orange', linestyle='--', label=f'Random Best: {random_best:.4f}')
for ev in unique_exploits:
    ax2.axvline(ev, color='green', linestyle=':', alpha=0.5)
ax2.set_title('PBT Population vs. Random Search')
ax2.set_xlabel('Epoche'); ax2.set_ylabel('Val-Accuracy')
ax2.legend(fontsize=9); ax2.grid(True)

# LR-Entwicklung über Zeit (für ein Individuum)
ax3 = axes[1, 0]
for i, (ind, color) in enumerate(zip(population[:5], colors)):
    ax3.plot(pbt_metrics[ind['id']]['lr_history'], color=color, alpha=0.8,
              label=f'Ind {ind["id"]}')
ax3.set_title('LR-Entwicklung (Top-5 Individuen)')
ax3.set_xlabel('Epoche'); ax3.set_ylabel('Learning Rate')
ax3.set_yscale('log'); ax3.legend(fontsize=8); ax3.grid(True)

# Finale Vergleich-Boxplot
ax4 = axes[1, 1]
ax4.boxplot([pbt_final, random_results], labels=['PBT', 'Random Search'])
ax4.scatter([1]*len(pbt_final), pbt_final, color='blue', alpha=0.5, s=50)
ax4.scatter([2]*len(random_results), random_results, color='red', alpha=0.5, s=50)
ax4.set_title('Finale Val-Accuracy: PBT vs. Random Search')
ax4.set_ylabel('Val-Accuracy'); ax4.grid(True, axis='y')

plt.tight_layout()
plt.savefig('E13_3_pbt_simulation.png', dpi=100)
plt.show()

print(f"\nPBT Verbesserung vs. Random: {(pbt_best - random_best)*100:.2f} Prozentpunkte (Best-to-Best)")
